# DeLong test: Fine-tuned DistilBERT vs Joint (DistilBERT + structured)

This notebook reproduces validation-set predicted probabilities from the **saved checkpoints** used in the project notebooks, then runs a paired **DeLong** test for the difference in ROC AUC.


In [5]:
import os
import math
import numpy as np
import pandas as pd
import torch

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import StandardScaler

# Repro paths (same as the other notebooks)
BERT_FT_CKPT = "./distilbert_results/checkpoint-16524"
JOINT_CKPT = "./joint_distilbert_structured_results/checkpoint-16524"

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)


device: cpu


In [6]:
def prepare_text_split(df: pd.DataFrame) -> pd.DataFrame:
    df = df[df["state"].isin(["successful", "failed"])].copy()
    df["text"] = df["name"].fillna("") + " " + df["blurb"].fillna("")
    df["label"] = (df["state"] == "successful").astype(int)
    df = df.dropna(subset=["text", "label"]).reset_index(drop=True)
    return df[["text", "label"]]


def load_feature_names(path: str):
    with open(path) as f:
        return [line.strip() for line in f if line.strip()]


FEATURES_NO_SCALE = load_feature_names("data/features/features_no_scale.txt")
FEATURES_SCALE = load_feature_names("data/features/features_scale.txt")


def build_tabular_matrix(df_aligned: pd.DataFrame, *, fit_on: pd.DataFrame):
    """Build the same 29-d structured vector (fit scaler on train only)."""
    X_no_train = fit_on[FEATURES_NO_SCALE].astype(np.float32).to_numpy()
    X_scale_train_raw = fit_on[FEATURES_SCALE].astype(np.float32).to_numpy()

    scaler = StandardScaler()
    X_scale_train = scaler.fit_transform(X_scale_train_raw)

    X_no = df_aligned[FEATURES_NO_SCALE].astype(np.float32).to_numpy()
    X_scale_raw = df_aligned[FEATURES_SCALE].astype(np.float32).to_numpy()
    X_scale = scaler.transform(X_scale_raw)

    X = np.hstack([X_no, X_scale]).astype(np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    return X


# Load raw splits
train_raw = pd.read_csv("data/features/train.csv")
val_raw = pd.read_csv("data/features/val.csv")

# Prepare text-only frames (also gives aligned filter)
train_text = prepare_text_split(train_raw)
val_text = prepare_text_split(val_raw)

# For the structured matrix we need the same row filtering as text split
# (keep all columns, but aligned rows only)
train_aligned = train_raw[train_raw["state"].isin(["successful", "failed"])].copy().reset_index(drop=True)
val_aligned = val_raw[val_raw["state"].isin(["successful", "failed"])].copy().reset_index(drop=True)

# Make sure label/text alignment matches
train_aligned["label"] = (train_aligned["state"] == "successful").astype(int)
val_aligned["label"] = (val_aligned["state"] == "successful").astype(int)
train_aligned["text"] = train_aligned["name"].fillna("") + " " + train_aligned["blurb"].fillna("")
val_aligned["text"] = val_aligned["name"].fillna("") + " " + val_aligned["blurb"].fillna("")

# Sanity: val_text is a subset view of val_aligned columns
assert len(val_text) == len(val_aligned)
assert np.all(val_text["label"].to_numpy() == val_aligned["label"].to_numpy())

X_train_tab = build_tabular_matrix(train_aligned, fit_on=train_aligned)
X_val_tab = build_tabular_matrix(val_aligned, fit_on=train_aligned)

y_val = val_text["label"].to_numpy()

print("val rows:", len(y_val))
print("tab dim:", X_val_tab.shape)


val rows: 22032
tab dim: (22032, 29)


In [7]:
def get_val_probs_bert_finetuned(texts):
    if not os.path.isdir(BERT_FT_CKPT):
        raise FileNotFoundError(f"Missing checkpoint dir: {BERT_FT_CKPT}")

    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = AutoModelForSequenceClassification.from_pretrained(BERT_FT_CKPT).to(DEVICE)

    ds = Dataset.from_pandas(pd.DataFrame({"text": texts}))

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

    ds = ds.map(tok, batched=True)
    ds = ds.remove_columns(["text"])
    ds.set_format("torch")

    args = TrainingArguments(
        output_dir="./_tmp_delong_bert_ft",
        per_device_eval_batch_size=32,
        report_to="none",
    )
    trainer = Trainer(model=model, args=args, tokenizer=tokenizer)

    pred = trainer.predict(ds)
    logits = pred.predictions
    probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()[:, 1]
    return probs


val_probs_bert_ft = get_val_probs_bert_finetuned(val_text["text"].tolist())
auc_bert_ft = float(roc_auc_score(y_val, val_probs_bert_ft))
print("AUC (fine-tuned DistilBERT, text-only):", auc_bert_ft)


Map:   0%|          | 0/22032 [00:00<?, ? examples/s]

/var/folders/0f/1mt731m54b51gldjgy_zszqr0000gn/T/ipykernel_25917/3179200332.py:22: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=args, tokenizer=tokenizer)


AUC (fine-tuned DistilBERT, text-only): 0.8390960707093926


In [8]:
import torch.nn as nn
from transformers import AutoModel


def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state * mask, dim=1)
    denom = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / denom


class DistilBERTWithStructuredFeatures(nn.Module):
    def __init__(self, model_name: str, structured_dim: int, dropout: float = 0.2, hidden_dim: int = 256):
        super().__init__()
        self.text_encoder = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Sequential(
            nn.Linear(768 + structured_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, 2),
        )
        self.loss_fn = nn.CrossEntropyLoss()

    def forward(self, input_ids=None, attention_mask=None, structured_features=None, labels=None):
        out = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        pooled = mean_pooling(out.last_hidden_state, attention_mask)
        combined = torch.cat([pooled, structured_features], dim=1)
        combined = self.dropout(combined)
        logits = self.classifier(combined)
        loss = None
        if labels is not None:
            loss = self.loss_fn(logits, labels)
        return {"loss": loss, "logits": logits}


def get_val_probs_joint(texts, X_tab):
    if not os.path.isdir(JOINT_CKPT):
        raise FileNotFoundError(f"Missing checkpoint dir: {JOINT_CKPT}")

    tokenizer = AutoTokenizer.from_pretrained("distilbert-base-uncased")
    model = DistilBERTWithStructuredFeatures(
        model_name="distilbert-base-uncased",
        structured_dim=X_tab.shape[1],
        dropout=0.2,
        hidden_dim=256,
    ).to(DEVICE)

    # Load weights saved by Trainer
    weights_safetensors = os.path.join(JOINT_CKPT, "model.safetensors")
    weights_bin = os.path.join(JOINT_CKPT, "pytorch_model.bin")
    if os.path.isfile(weights_safetensors):
        from safetensors.torch import load_file

        state = load_file(weights_safetensors)
    elif os.path.isfile(weights_bin):
        state = torch.load(weights_bin, map_location="cpu", weights_only=True)
    else:
        raise FileNotFoundError(f"No model.safetensors or pytorch_model.bin in {JOINT_CKPT!r}")

    model.load_state_dict(state, strict=True)

    df = pd.DataFrame({"text": texts})
    ds = Dataset.from_pandas(df)

    def tok(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=128)

    ds = ds.map(tok, batched=True)
    ds = ds.remove_columns(["text"])

    # Add structured features (store as plain Python lists so HF Datasets can tensorize cleanly)
    ds = ds.add_column("structured_features", X_tab.astype(np.float32).tolist())

    # Match `joint_finetuning.ipynb`: include labels so the model produces a real loss (not None)
    # and Trainer can gather outputs without hitting `_pad_across_processes` NoneType.
    ds = ds.add_column("labels", val_aligned["label"].astype(int).tolist())

    ds.set_format("torch")

    args = TrainingArguments(
        output_dir="./_tmp_delong_joint",
        per_device_eval_batch_size=16,
        report_to="none",
    )
    trainer = Trainer(model=model, args=args, tokenizer=tokenizer)

    pred = trainer.predict(ds)
    logits = pred.predictions
    probs = torch.softmax(torch.from_numpy(logits), dim=-1).numpy()[:, 1]
    return probs


val_probs_joint = get_val_probs_joint(val_aligned["text"].tolist(), X_val_tab)
auc_joint = float(roc_auc_score(y_val, val_probs_joint))
print("AUC (joint fine-tuning: DistilBERT + structured):", auc_joint)


Map:   0%|          | 0/22032 [00:00<?, ? examples/s]

/var/folders/0f/1mt731m54b51gldjgy_zszqr0000gn/T/ipykernel_25917/2030073125.py:86: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(model=model, args=args, tokenizer=tokenizer)


AUC (joint fine-tuning: DistilBERT + structured): 0.857012501422211


In [10]:
import math
import numpy as np


def _compute_midrank(x):
    x = np.asarray(x)
    J = np.argsort(x)
    Z = x[J]
    N = len(x)
    T = np.zeros(N, dtype=float)

    i = 0
    while i < N:
        j = i
        while j < N and Z[j] == Z[i]:
            j += 1
        T[i:j] = 0.5 * (i + j - 1) + 1
        i = j

    out = np.empty(N, dtype=float)
    out[J] = T
    return out


def _fast_delong(predictions_sorted_transposed, label_1_count):
    """
    predictions_sorted_transposed: shape (n_classifiers, n_examples)
    with positive examples first.
    """
    m = label_1_count
    n = predictions_sorted_transposed.shape[1] - m
    k = predictions_sorted_transposed.shape[0]

    positive_examples = predictions_sorted_transposed[:, :m]
    negative_examples = predictions_sorted_transposed[:, m:]

    tx = np.zeros((k, m), dtype=float)
    ty = np.zeros((k, n), dtype=float)
    tz = np.zeros((k, m + n), dtype=float)

    for r in range(k):
        tx[r] = _compute_midrank(positive_examples[r])
        ty[r] = _compute_midrank(negative_examples[r])
        tz[r] = _compute_midrank(predictions_sorted_transposed[r])

    aucs = tz[:, :m].sum(axis=1) / m / n - (m + 1.0) / (2.0 * n)

    v01 = (tz[:, :m] - tx) / n
    v10 = 1.0 - (tz[:, m:] - ty) / m

    sx = np.cov(v01)
    sy = np.cov(v10)
    delongcov = sx / m + sy / n

    return aucs, delongcov


def delong_aucs_and_covariance(y_true, pred_mat):
    y_true = np.asarray(y_true).astype(int)
    order = np.argsort(-y_true)   # positives first
    y_sorted = y_true[order]
    m = int(y_sorted.sum())

    preds_sorted = pred_mat[:, order]
    aucs, cov = _fast_delong(preds_sorted, m)
    return aucs, cov


def normal_cdf(x):
    return 0.5 * (1.0 + math.erf(x / math.sqrt(2.0)))


def delong_pvalue_and_cis(y_true, probs_a, probs_b):
    preds = np.vstack([probs_a, probs_b])
    aucs, cov = delong_aucs_and_covariance(y_true, preds)

    auc_a, auc_b = float(aucs[0]), float(aucs[1])
    diff = auc_b - auc_a

    var_a = float(cov[0, 0])
    var_b = float(cov[1, 1])
    cov_ab = float(cov[0, 1])

    var_diff = var_a + var_b - 2.0 * cov_ab
    se_diff = math.sqrt(max(var_diff, 1e-15))

    z = diff / se_diff
    p = 2.0 * (1.0 - normal_cdf(abs(z)))

    def ci(zcrit):
        return (diff - zcrit * se_diff, diff + zcrit * se_diff)

    ci_95 = ci(1.959963984540054)
    ci_90 = ci(1.6448536269514722)

    se_a = math.sqrt(max(var_a, 1e-15))
    se_b = math.sqrt(max(var_b, 1e-15))
    auc_a_ci95 = (auc_a - 1.959963984540054 * se_a, auc_a + 1.959963984540054 * se_a)
    auc_b_ci95 = (auc_b - 1.959963984540054 * se_b, auc_b + 1.959963984540054 * se_b)

    return {
        "auc_a": auc_a,
        "auc_b": auc_b,
        "diff_b_minus_a": diff,
        "z": z,
        "p_value": p,
        "ci_95_diff": ci_95,
        "ci_90_diff": ci_90,
        "auc_a_ci95": auc_a_ci95,
        "auc_b_ci95": auc_b_ci95,
        "delong_cov": cov,
    }

In [11]:
res = delong_pvalue_and_cis(y_val, val_probs_bert_ft, val_probs_joint)

print("\n=== Paired DeLong test (AUC difference) ===")
print("AUC A (fine-tuned text-only):", res["auc_a"])
print("AUC B (joint + structured):", res["auc_b"])
print("Diff (B - A):", res["diff_b_minus_a"])
print("z:", res["z"])
print("p-value:", res["p_value"])
print("90% CI for diff:", res["ci_90_diff"])
print("95% CI for diff:", res["ci_95_diff"])


=== Paired DeLong test (AUC difference) ===
AUC A (fine-tuned text-only): 0.8390960707093925
AUC B (joint + structured): 0.8570125014222109
Diff (B - A): 0.017916430712818343
z: 16.264505084445123
p-value: 0.0
90% CI for diff: (0.01610451538634535, 0.019728346039291335)
95% CI for diff: (0.015757400435148554, 0.020075460990488132)
